In [2]:
import pandas as pd  # 用于数据处理和分析的库，这里用于读取和处理CSV格式的推文数据集
import torch  # PyTorch深度学习框架，用于构建和训练神经网络模型
import torch.nn as nn  # PyTorch的神经网络模块，包含各种层和损失函数
import numpy as np  # 用于科学计算的库，提供高效的数组操作
from transformers import DistilBertTokenizerFast, DistilBertModel, DistilBertForSequenceClassification  # 从Hugging Face的transformers库导入DistilBERT相关组件：快速分词器、基础模型和用于序列分类的预训练模型

# 从CSV文件加载清洗后的COVID-19情感分析数据集，该文件位于项目根目录下
dataset = pd.read_csv("../../clean_COVIDSenti.csv")

# 加载预训练的DistilBERT分词器（使用uncased版本，即不区分大小写）
# DistilBERT是BERT的轻量级版本，通过知识蒸馏技术压缩，速度更快且参数量更少
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

# 定义分词函数，将原始推文文本转换为模型可接受的token格式
def tokenize(tweet):
    # 对推文进行分词处理，设置以下参数：
    # - return_tensors='pt': 返回PyTorch张量格式
    # - padding="max_length": 将所有序列填充到最大长度
    # - max_length=47: 设置最大长度为47，这是基于COVID-19推文数据集的最大token长度统计结果
    tokenized = tokenizer(tweet, return_tensors='pt', padding="max_length", max_length = 47)
    return tokenized

# 提取推文文本和对应的标签，并将标签值加1使其变为从0开始的索引（原始标签可能是-1,0,1，需要转换为0,1,2）
tweets, labels = dataset['tweet'], dataset['label'] + 1

# 对所有推文进行批量分词处理，将文本转换为token序列
tokenized_tweets = tweets.map(tokenize)

# 将分词后的推文和标签转换为列表格式，便于后续创建数据集对象
tokenized_tweets, labels = tokenized_tweets.to_list(), labels.to_list()

# 定义设备选择函数，自动检测并使用可用的最佳计算设备（GPU/NPU/CPU）
def get_device():
    # 首先检查Intel XPU（Intel的GPU加速方案）
    try:
        import intel_extension_for_pytorch as ipex  # Intel为PyTorch提供的扩展库，用于Intel硬件加速
        if hasattr(torch, 'xpu') and torch.xpu.is_available():
            device = torch.device("xpu")
            print(f"Training on Intel XPU: {torch.xpu.get_device_name(0)}")
            return device
    except (ImportError, OSError) as e:
        print(f"Intel XPU not available: {e}")
        pass
    
    # 检查Apple MPS（Metal Performance Shaders，适用于Apple Silicon芯片的GPU加速）
    if torch.backends.mps.is_available():
        device = torch.device("mps")
        print("Training on Apple GPU (MPS)")
        return device
    
    # 检查NVIDIA CUDA（NVIDIA GPU的并行计算平台）
    if torch.cuda.is_available():
        device = torch.device("cuda")
        print(f"Training on NVIDIA CUDA: {torch.cuda.get_device_name(0)}")
        return device
    
    # 检查OpenVINO NPU（Intel的神经网络处理单元）
    try:
        import openvino as ov  # Intel的深度学习推理优化工具包
        core = ov.Core()
        available_devices = core.available_devices
        if 'NPU' in available_devices:
            print(f"OpenVINO NPU available: {core.get_property('NPU', 'DEVICE_FULL_NAME')}")
    except:
        pass
    
    # 如果以上加速设备都不可用，则回退到CPU进行训练
    print("Training on CPU")
    return torch.device("cpu")

# 获取当前可用的计算设备
device = get_device()
# 打印PyTorch版本信息，便于调试和复现
print(f"PyTorch version: {torch.__version__}")
# 打印当前使用的设备信息
print(f"Device: {device}")

Intel XPU not available: No module named 'intel_extension_for_pytorch'
Training on CPU
PyTorch version: 2.4.1+cpu
Device: cpu


In [4]:
from scipy.stats import poisson  # 导入泊松分布模块，用于生成随机迭代次数，实现"思考"机制
import inspect  # 用于检查函数签名和参数信息的模块

class BertWithThinking(nn.Module):
    """
    实现一个带有"思考"机制的BERT变体模型。
    该模型通过泊松分布随机决定某个Transformer层的迭代次数，
    模拟人类在做出判断前的"反复思考"过程。
    当连续两次迭代的输出差异小于阈值时，提前退出循环。
    """
    
    def __init__(self, init_bert, poisson_mean):
        """
        初始化BertWithThinking模型
        
        参数:
            init_bert: 预训练的DistilBERT模型实例
            poisson_mean: 泊松分布的均值参数，控制平均迭代次数
        """
        super().__init__()
        self.bert = init_bert  # 保存预训练的BERT模型引用
        self.poisson_mean = poisson_mean  # 存储泊松分布均值参数
        self.embedding = self.bert.distilbert.embeddings  # 提取词嵌入层，用于将token IDs转换为向量表示
        
        # 定义"前奏"层（prelude）：处理输入的前4个Transformer层（索引0-3）
        # 这些层负责提取基础的语义特征
        self.prelude = nn.ModuleList([
            self.bert.distilbert.transformer.layer[0-4]  # 注意：这里的切片语法可能有误，应该是[:4]
        ])
        
        # 定义"循环"层（recurrent）：第5个Transformer层（索引4）
        # 这是实现"思考"机制的核心层，会根据泊松分布重复执行多次
        self.recurrent = self.bert.distilbert.transformer.layer[4]
        
        # 定义"尾声"层（coda）：最后一个Transformer层（索引5）
        # 在循环层之后执行，用于进一步处理特征
        self.coda = nn.Sequential(
            self.bert.distilbert.transformer.layer[5]
        )
        
        # 定义分类器模块，包含预分类层、分类层和dropout层
        # pre_classifier: 将序列输出聚合为固定维度的表示
        # classifier: 最终的分类层，输出各类别的logits
        # dropout: 防止过拟合的正则化层
        self.classifier = nn.Sequential(
            self.bert.pre_classifier, 
            self.bert.classifier, 
            self.bert.dropout
        )
        
    def forward(self, input_ids, attention_mask):
        """
        前向传播函数，定义数据流经模型的路径
        
        参数:
            input_ids: 输入文本的token ID序列
            attention_mask: 注意力掩码，标识哪些位置是有效token（1）vs 填充（0）
            
        返回:
            分类logits，形状为(batch_size, num_labels)
        """
        # 步骤1：将输入的token IDs通过嵌入层转换为稠密向量表示
        # 嵌入层会结合token embedding、position embedding和segment embedding
        output = self.bert.distilbert.embeddings(input_ids=input_ids)
        
        # 步骤2：通过前4个Transformer层（prelude）处理嵌入向量
        # 这些层执行标准的自注意力计算和前馈网络操作
        for layer in self.bert.distilbert.transformer.layer[:4]:
            # 打印层的函数签名信息，用于调试（生产环境应移除）
            print(inspect.getfullargspec(layer.forward))
            # 将当前层的输出和注意力掩码传递给下一个Transformer层
            output = layer(output, attn_mask=attention_mask)
        
        # 步骤3：保存循环层之前的输出，用于后续的收敛性判断
        prev_output = output
        
        # 根据泊松分布随机采样得到迭代次数
        # poisson.rvs返回随机变量样本，size=1表示采样1次
        recurrences = poisson.rvs(self.poisson_mean, size=1)
        
        # 步骤4：执行"思考"循环 - 重复通过第5个Transformer层
        # 模拟人类反复思考的过程，直到输出稳定或达到最大迭代次数
        for _ in range(recurrences[0]):
            # 通过循环层（第5个Transformer层）处理当前输出
            output = self.bert.distilbert.transformer.layer[4](output, attn_mask=attention_mask)
            
            # 计算当前输出与上一次输出之间的L2范数差异
            # torch.norm(output - prev_output, p=2, dim=-1)计算每个token向量的L2距离
            # torch.mean对所有token的差异取平均，得到标量值
            diff = torch.mean(torch.norm(output - prev_output, p=2, dim=-1))
            
            # 如果输出变化小于退出阈值，说明模型已经"想清楚了"，提前终止循环
            # 这是一种动态计算策略，可以提高效率并模拟人类的思考过程
            if diff < self.exit_threshold:
                break
            
            # 更新前一次输出为当前输出，用于下一次迭代的比较
            prev_output = output
        
        # 步骤5：通过最后一个Transformer层（coda，索引5）处理循环层的最终输出
        # 这一步对"思考"后的特征进行最后的提炼
        output = self.bert.distilbert.transformer.layer[5](output, attn_mask=attention_mask)
        
        # 步骤6：通过分类器模块进行最终的情感分类
        # pre_classifier: 将[CLS] token的输出映射到隐藏维度
        output = self.bert.pre_classifier(output)
        # classifier: 将隐藏表示映射到类别logits（3个情感类别）
        output = self.bert.classifier(output)
        # dropout: 应用dropout正则化，防止过拟合
        output = self.bert.dropout(output)
        
        return output

In [5]:
class IterativeLayer(nn.Module):
    """
    可迭代的Transformer层封装器，实现动态计算深度的"思考"机制。
    该类将单个Transformer层包装成可以重复执行多次的模块，
    通过泊松分布决定迭代次数，并通过输出收敛性判断提前退出。
    """
    
    def __init__(self, init_layer, poisson_mean, exit_threshold=1e-4):
        """
        初始化IterativeLayer
        
        参数:
            init_layer: 要包装的原始Transformer层实例
            poisson_mean: 泊松分布的均值参数，控制平均迭代次数
            exit_threshold: 退出阈值，当连续两次输出的L2距离小于此值时提前退出循环
        """
        super().__init__()
        self.layer = init_layer  # 保存原始Transformer层的引用
        self.poisson_mean = poisson_mean  # 存储泊松分布均值参数
        self.exit_threshold = exit_threshold  # 存储收敛性判断阈值

    def forward(self, hidden_state, mask, *args, **kwargs):
        """
        前向传播函数，执行迭代计算
        
        参数:
            hidden_state: 输入的隐藏状态张量，形状为(batch_size, seq_len, hidden_dim)
            mask: 注意力掩码，标识有效token位置
            *args, **kwargs: 其他可能传递给原始层的参数
            
        返回:
            经过迭代处理后的隐藏状态张量
        """
        # 保存初始隐藏状态，用于第一次迭代的收敛性比较
        prev_state = hidden_state
        
        # 从泊松分布中采样得到本次前向传播的迭代次数
        # poisson.rvs返回数组，[0]取出标量值
        iterations = poisson.rvs(self.poisson_mean, size=1)[0]
        
        # 执行迭代计算循环
        for _ in range(iterations):
            # 通过原始Transformer层处理当前隐藏状态
            # attn_mask参数用于屏蔽填充位置的注意力计算
            hidden_state = self.layer(hidden_state, attn_mask=mask)
            
            # 计算当前隐藏状态与上一次隐藏状态之间的L2范数差异
            # hidden_state[0]表示取第一个元素（可能是tuple或单个张量）
            # torch.norm(..., p=2, dim=-1)计算最后一个维度上的L2距离
            # torch.mean对所有token位置取平均，得到标量差异值
            diff = torch.mean(torch.norm(hidden_state[0] - prev_state[0], p=2, dim=-1))
            
            # 如果输出变化小于阈值，说明层输出已经收敛，提前退出循环
            # 这是一种自适应计算策略，可以根据输入复杂度动态调整计算量
            if diff < self.exit_threshold:
                break
            
            # 更新前一次状态为当前状态，用于下一次迭代的比较
            prev_state = hidden_state
            
        return hidden_state

In [ ]:
from torch.utils.data import DataLoader, WeightedRandomSampler, Dataset, random_split  # 导入PyTorch数据处理工具：DataLoader用于批量加载数据，WeightedRandomSampler用于处理类别不平衡，Dataset是数据集基类，random_split用于数据集划分

class TweetDataset(Dataset):
    """
    自定义推文数据集类，继承自PyTorch的Dataset基类。
    用于将分词后的推文数据和标签封装成PyTorch可使用的数据集格式。
    """
    
    def __init__(self, tweets, labels):
        """
        初始化数据集
        
        参数:
            tweets: 分词后的推文数据列表，每个元素是包含input_ids和attention_mask的字典
            labels: 情感标签列表（0, 1, 2分别表示负面、中性、正面）
        """
        self.x = tweets  # 存储推文特征数据
        self.y = labels  # 存储对应的标签数据
        
    def __getitem__(self, index):
        """
        根据索引获取单个样本，这是Dataset类必须实现的方法
        
        参数:
            index: 要获取的样本索引
            
        返回:
            包含特征字典和标签的元组 (x, y)
        """
        # 根据索引获取推文特征数据
        x = self.x[index]
        # 确保数据是字典格式（可能从其他格式转换而来）
        x = dict(x)
        # 对字典中的每个张量执行squeeze操作，移除维度为1的维度
        # 这是因为分词时可能产生了多余的批次维度
        x = {key: torch.squeeze(val, dim = 0) for key, val in x.items()}
        # 获取对应的标签
        y = self.y[index]
        return (x, y)
    
    def __len__(self):
        """
        返回数据集的总样本数，这是Dataset类必须实现的方法
        
        返回:
            数据集的样本总数
        """
        return len(self.x)
    
folds = 5  # 设置交叉验证的折数，5折交叉验证是常用的选择
early_stopping = 5  # 设置早停耐心值：当验证集准确率在连续5个epoch内没有提升时停止训练，这是一种正则化技术，防止过拟合并节省训练时间
train_frac = 0.8  # 设置数据集划分比例：80%训练集
test_frac = 0.1  # 10%测试集
val_frac = 0.1  # 10%验证集
batch_size = 64  # 设置批次大小，决定每次梯度更新使用的样本数，较大的batch_size可以提高训练速度但可能影响泛化性能
test_accuracies = []  # 用于存储每个fold的测试准确率，便于后续计算平均性能
print(isinstance(labels,pd.Series))  # 检查标签是否为pandas Series类型（调试用）
data = TweetDataset(tokenized_tweets, labels)  # 创建自定义数据集实例

# 开始5折交叉验证循环
for fold in range(folds):
    print(f"FOLD {fold}")
    
    # 创建随机数生成器并设置种子，确保每次fold的数据划分是可复现的
    gen = torch.Generator().manual_seed(fold)
    
    # 根据设定的比例随机划分数据集为训练集、验证集和测试集，使用相同的随机数生成器确保划分的可复现性
    train, val, test = random_split(data, lengths=[train_frac, val_frac, test_frac], generator=gen)
    
    # 处理训练集的类别不平衡问题：首先提取训练集中所有样本的标签
    labels_for_counts = list(map(lambda x: x[-1], train))
    # 计算每个类别频率的倒数作为类别权重（少数类别获得更高权重）
    frequency = 1 / np.bincount(labels_for_counts)
    class_weights = torch.tensor(frequency, dtype=torch.float32)
    # 为每个训练样本分配对应的类别权重
    obs_weights = list(map(lambda x: class_weights[x[-1]], train))
        
    # 注释掉的代码：使用加权随机采样器处理类别不平衡
    # train_sampler = WeightedRandomSampler(weights = obs_weights, num_samples = len(obs_weights))
    
    # 创建训练集数据加载器，启用shuffle以打乱数据顺序，打乱数据有助于提高模型的泛化能力
    train_loader = DataLoader(train, batch_size=batch_size, shuffle = True)
    
    # 创建验证集数据加载器，不shuffle以保持数据顺序
    val_loader = DataLoader(val, shuffle=False, batch_size=batch_size)
    
    # 创建测试集数据加载器，不shuffle以保持数据顺序
    test_loader = DataLoader(test, shuffle=False, batch_size=batch_size)
    
    # ---- 从这里开始训练实际模型 ----
    
    # 加载预训练的DistilBERT序列分类模型，设置输出类别数为3（负面、中性、正面）
    init_bert = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=3)
    
    # 将预训练模型赋值给model变量
    model = init_bert
    
    # 将模型的第5个Transformer层（索引4）替换为IterativeLayer，这使得该层可以执行多次迭代，实现"思考"机制，poisson_mean=3表示平均迭代3次
    model.distilbert.transformer.layer[4] = IterativeLayer(model.distilbert.transformer.layer[4], poisson_mean=3)
    
    # 注释掉的代码：使用BertWithThinking模型（另一种实现方式）
    # model = BertWithThinking(init_bert = init_bert, poisson_mean=3)
    
    # 将模型移动到之前检测到的计算设备（CPU/GPU/NPU）
    model = model.to(device)

    # 指定需要训练的层：预分类层、分类层、第5个Transformer层（迭代层）、第6个Transformer层，这种部分训练策略可以节省大量训练时间，同时保持模型性能
    trained_layers = [model.pre_classifier, model.classifier, model.distilbert.transformer.layer[4], model.distilbert.transformer.layer[5]]
    
    # 创建参数列表用于收集需要训练的层参数
    updated_params = nn.ParameterList([])
    for layer in trained_layers:
        updated_params.extend(layer.parameters())

    # 设置所有模型参数为可训练状态，注释说明：通常这里会设置为False以冻结预训练层，但当前策略是训练整个模型
    for param in model.parameters():
        param.requires_grad = True

    # 将模型设置为训练模式，启用dropout等训练特定的行为
    model.train()

    # 设置学习率，较小的学习率有助于稳定训练过程
    lr = 0.0001
    
    # 初始化训练轮次计数器
    epoch = 0
    
    # 初始化早停计数器，记录验证集准确率没有提升的连续epoch数
    no_improvement = 0
    
    # 初始化当前最佳验证集准确率
    curr_acc = 0
    
    # 定义损失函数为交叉熵损失，适用于多分类任务，注释说明：不使用softmax是因为CrossEntropyLoss内部已经包含了softmax操作
    criterion = nn.CrossEntropyLoss()
    
    # 定义优化器为AdamW，这是一种带有权重衰减的Adam优化器，AdamW在Transformer模型上通常表现良好
    optimizer = torch.optim.AdamW(model.parameters(), lr = lr)

    # 开始训练循环，直到触发早停条件
    while no_improvement < early_stopping:
        epoch += 1
        print(f"Epoch {epoch}")
        
        # ---- 训练阶段 ----
        # 遍历训练数据加载器中的每个批次
        for train_inputs, train_labels in train_loader:
            # 将输入特征（input_ids和attention_mask）移动到计算设备
            train_inputs['input_ids'], train_inputs['attention_mask'] = train_inputs['input_ids'].to(device), train_inputs['attention_mask'].to(device)
            # 将标签移动到计算设备
            train_labels = train_labels.to(device)
            
            # 清除上一轮迭代的梯度
            model.zero_grad()
            
            # 使用自动混合精度训练（针对MPS设备），混合精度可以加速训练并减少内存使用
            with torch.autocast("mps"):
                # 前向传播：将输入数据传入模型，获取logits输出
                output = model(**train_inputs)['logits']   
            
            # 计算预测输出与真实标签之间的交叉熵损失
            loss = criterion(output, train_labels)
            
            # 反向传播：计算损失相对于模型参数的梯度
            loss.backward()
            
            # 优化器更新：根据计算出的梯度更新模型参数
            optimizer.step()
        
        # ---- 验证阶段（早停机制） ----
        # 将模型设置为评估模式，禁用dropout等训练特定行为
        model.eval()
        
        # 初始化正确和错误预测的计数器
        correct = torch.tensor(0, device = device)
        incorrect = torch.tensor(0, device = device)
        
        # 遍历验证数据加载器中的每个批次
        for val_inputs, val_labels in val_loader:
            # 将验证集输入特征移动到计算设备
            val_inputs['input_ids'], val_inputs['attention_mask'] = val_inputs['input_ids'].to(device), val_inputs['attention_mask'].to(device)
            # 将验证集标签移动到计算设备
            val_labels = val_labels.to(device)
            
            # 前向传播：获取验证集的预测logits
            probs = model(**val_inputs)['logits']
            
            # 取logits最大值的索引作为预测类别（argmax操作）
            preds = torch.argmax(probs, axis = 1)
            
            # 确保预测结果在正确的计算设备上
            preds = preds.to(device)
            
            # 统计正确预测的数量
            correct += (preds == val_labels).sum()
            # 统计错误预测的数量
            incorrect += (preds != val_labels).sum()  
        
        # 计算验证集准确率
        accuracy = correct / (correct + incorrect)
        
        # 检查是否有性能提升
        if accuracy > curr_acc:
            # 如果当前准确率超过历史最佳，更新最佳准确率并重置早停计数器
            print(f"New accuracy has been reached: {accuracy}")
            curr_acc = accuracy
            no_improvement = 0
        else:
            # 如果没有提升，增加早停计数器
            no_improvement += 1
        
        # 将模型重新设置为训练模式，准备下一轮训练
        model.train()
        
    # ---- 测试阶段 ----
    # 将模型设置为评估模式
    model.eval()
    
    # 初始化测试集正确和错误预测的计数器
    correct = torch.tensor(0, device = device)
    incorrect = torch.tensor(0, device = device)
    
    # 获取测试集准确率用于交叉验证评估
    for test_inputs, test_labels in test_loader:
        # 将测试集输入特征移动到计算设备
        test_inputs['input_ids'], test_inputs['attention_mask'] = test_inputs['input_ids'].to(device), test_inputs['attention_mask'].to(device)
        # 将测试集标签移动到计算设备
        test_labels = test_labels.to(device)
        
        # 前向传播：获取测试集的预测logits
        probs = model(**test_inputs)['logits']
        
        # 取logits最大值的索引作为预测类别
        preds = torch.argmax(probs, axis = 1)
        
        # 确保预测结果在正确的计算设备上
        preds = preds.to(device)
        
        # 统计正确预测的数量
        correct += (preds == test_labels).sum()
        # 统计错误预测的数量
        incorrect += (preds != test_labels).sum()  
    
    # 计算当前fold的测试准确率
    test_accuracy = correct / (correct + incorrect)
    
    # 将当前fold的测试准确率添加到列表中
    test_accuracies.append(test_accuracy)
    
    # 打印当前fold的测试结果
    print(f"FOR FOLD {fold}, THE TEST ACCURACY WAS {test_accuracy}")
    print("---------------------------------------")
        



False
FOLD 0


c:\Users\manaf\miniconda3\envs\covid-sentiment\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\manaf\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
